In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange

sys.path.append("../../")
import biked_commons
from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.resource_utils import split_datasets_path
from biked_commons.conditioning import conditioning
from biked_commons.design_evaluation.scoring import *

/home/mileva/mambaforge/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")  

data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

data_tens = torch.tensor(data.values, dtype=torch.float32, device=device)


In [3]:
evaluator, requirement_names, requirement_types = construct_tensor_evaluator(get_standard_evaluations(device), data.columns)

In [4]:
# num_data = data.shape[0]
num_data = 1
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")
image_embeddings = conditioning.sample_image_embedding(num_data, split="test")
condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Embedding": image_embeddings}
# condition = {"Rider": rider_condition, "Use Case": use_case_condition, "Text": text_condition}

In [15]:
requirement_names

['Mass',
 'Planar Compliance',
 'Transverse Compliance',
 'Eccentric Compliance',
 'Planar Safety Factor',
 'Eccentric Safety Factor']

In [121]:
#calcualte gradient of scores wrt data_tens

data_tens.requires_grad = True
data_tens.grad = None
eval_scores = evaluator(data_tens, condition)
score_sum = eval_scores[:, 4].sum()
score_sum.backward()


In [122]:
#check for infs and nans in data_tens
if torch.any(torch.isnan(data_tens)) or torch.any(torch.isinf(data_tens)):
    print("Data tensor contains NaN or Inf values.")

In [123]:
#get nan indices of data_tens.grad
nan_indices = torch.isnan(data_tens.grad).nonzero(as_tuple=True)
print("nan indices: ", nan_indices)

nan indices:  (tensor([], device='cuda:0', dtype=torch.int64), tensor([], device='cuda:0', dtype=torch.int64))


In [124]:
grad_df = pd.DataFrame(data_tens.grad.cpu().detach().numpy(), columns=data.columns)
thickness_grads = grad_df[["Wall thickness Down tube", "Wall thickness Bottom Bracket", "Wall thickness Seat tube", "Wall thickness Top tube", "Wall thickness Chain stay", "Wall thickness Seat stay", "Wall thickness Head tube"]]
thickness_grads.mean(axis=0)

Wall thickness Down tube        -0.100656
Wall thickness Bottom Bracket   -0.015436
Wall thickness Seat tube        -0.047320
Wall thickness Top tube         -0.051658
Wall thickness Chain stay       -0.004797
Wall thickness Seat stay        -0.005400
Wall thickness Head tube        -0.001284
dtype: float32

In [9]:
isobjective = torch.tensor(requirement_types) == 1
isobjective = isobjective.to(device)
objective_scores = eval_scores[:, isobjective].detach().cpu().numpy()
# constraint_scores = eval_scores[:, ~isobjective].detach().numpy()

In [10]:
main_scorer = construct_scorer(MainScores, get_standard_evaluations(device), data.columns, device)
detailed_scorer = construct_scorer(DetailedScores, get_standard_evaluations(device), data.columns, device)

In [11]:
main_scorer(data_tens.detach(), condition)

Hypervolume                     0.917943
Constraint Satisfaction Rate    0.006892
Maximum Mean Discrepancy        0.000000
dtype: float64

In [12]:
detailed_scorer(data_tens.detach(), condition)

Min Objective Score: Mass                                       2.001677
Min Objective Score: Planar Compliance                          0.000000
Min Objective Score: Transverse Compliance                      0.000000
Min Objective Score: Eccentric Compliance                       0.000000
Mean Objective Score: Mass                                      6.937059
Mean Objective Score: Planar Compliance                         0.226122
Mean Objective Score: Transverse Compliance                     0.614752
Mean Objective Score: Eccentric Compliance                      0.640393
Constraint Violation Rate: Planar Safety Factor                 0.988439
Constraint Violation Rate: Eccentric Safety Factor              0.953980
Mean Constraint Violation Magnitude: Planar Safety Factor       1.021764
Mean Constraint Violation Magnitude: Eccentric Safety Factor    0.806215
dtype: float64